# [IAPR][iapr]: Lab 3 ‒  Classification


**Group ID:** xx

**Author 1 (sciper):** Student Name 1 (xxxxx)  
**Author 2 (sciper):** Student Name 2 (xxxxx)   
**Author 3 (sciper):** Student Name 3 (xxxxx)   

**Release date:** 19.04.2023  
**Due date:** 05.05.2023 


## Important notes

The lab assignments are designed to teach practical implementation of the topics presented during class well as
preparation for the final project, which is a practical project which ties together the topics of the course.

As such, in the lab assignments/final project, unless otherwise specified, you may, if you choose, use external
functions from image processing/ML libraries like opencv and sklearn as long as there is sufficient explanation
in the lab report. For example, you do not need to implement your own edge detector, etc.

**! Before handling back the notebook <font color='red'> rerun </font>the notebook from scratch !**
`Kernel` > `Restart & Run All`

We will not rerun the notebook for you.


[iapr]: https://github.com/LTS5/iapr

--
## 0. Setup

In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
!python3 --version

Python 3.7.16


In this lab, we will use PyTorch. If you are not familiar with this library, [here](https://pytorch.org/tutorials/beginner/blitz/tensor_tutorial.html) is a quick tutorial of the basics.

In [22]:
import platform
print(platform.system())
if platform.system() == "Darwin":
    %pip install torch==1.8.1 torchvision==0.9.1
else:
    %pip install torch==1.8.1+cu111 torchvision==0.9.1+cu111 -f https://download.pytorch.org/whl/torch_stable.html

Linux
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://packagecloud.io/github/git-lfs/pypi/simple
Looking in links: https://download.pytorch.org/whl/torch_stable.html
DEPRECATION: The HTML index page being used (https://download.pytorch.org/whl/torch_stable.html) is not a proper HTML 5 document. This is in violation of PEP 503 which requires these pages to be well-formed HTML 5 documents. Please reach out to the owners of this index page, and ask them to update this index page to a valid HTML 5 document. pip 22.2 will enforce this behaviour change. Discussion can be found at https://github.com/pypa/pip/issues/10825
ERROR: Could not find a version that satisfies the requirement torch==1.8.1+cu111 (from versions: 1.11.0, 1.11.0+cpu, 1.11.0+cu102, 1.11.0+cu113, 1.11.0+cu115, 1.11.0+rocm4.3.1, 1.11.0+rocm4.5.2, 1.12.0, 1.12.0+cpu, 1.12.0+cu102, 1.12.0+cu113, 1.12.0+cu116, 1.12.0+rocm5.0, 1.12.0+rocm5.1.1, 1.1

In [23]:
import tarfile
import os

data_base_path = os.path.join(os.pardir, 'data')
data_folder = 'lab-03-data'
tar_path = os.path.join(data_base_path, data_folder + '.tar.gz')
with tarfile.open(tar_path, mode='r:gz') as tar:
    tar.extractall(path=data_base_path)

---
## Part 1 - Out-of-Distribution detection in colorectal cancer histology (12 points)

Colorectal cancer is one of the most widespread cancers for men and women. Diagnosis complemented with prognostic and predictive biomarker information is essential for patient monitoring and applying personalized treatments. A critical marker is the tumor/stroma ratio in unhealthy tissues sampled from the colon. The higher the ratio, the more invasive the cancer is. The degree of invasion is tightly linked to patient survial probability.

To measure the ratio, a pathologist needs to analyze the unhealthy tissue under a microscope and estimate it from a look. As the number of samples to analyze is huge and estimations are only sometimes precise, automatic recognition of the different tissue types in histological images has become essential. Such an automatic process requires the development of a multi-class classifier to identify the numerous tissues. As shown below, they are usually 8 tissue types to categorize: TUMOR, STROMA, LYMPHO (lymphocytes), MUCOSA, COMPLEX (complex stroma), DEBRIS, ADIPOSE and EMPTY (background).

<br />
<br />
<figure>
    <img src="../data/lab-03-data/part1/kather16.svg" width="1100">
    <center>
    <figcaption>Fig1: Collection of tissue types in colorectal cancer histology (Kather-16)</figcaption>
    </center>
</figure>
<br />
<br />


Up to this day, state-of-the-art methods use deep-learning-based supervised learning methods. A downfall of such an approach is the necessity to access a well-annotated training dataset. In histology, annotating data is difficult. It is time-consuming and requires the expertise of pathologists. Moreover, the annotator must label every tissue type while only two (TUMOR and STROMA) are interesting. 


Consequently, we propose another approach. In order to make the annotation task less tedious, we ask the annotator to label only the tissues of interest and dump the others. Then, we must train a binary classifier to automatically recognize these tissues at test time. In this part, you will implement the proposed approach.

### 1.1 Binary classifier with Mahalanobis distance (3 points)

Based on the abovementioned process, your task is to build a model that recognizes TUMOR (Label 0) and STROMA (Label 1) tissue types. Your model will be supervised by a training dataset containing TUMOR and STROMA annotations; note that all other tissues have been dropped.
We will not ask you to train a deep-learning-based binary classifier from scratch. Instead, we provide excellent features (descriptors) of the images we extracted from a visual foundation model. (Note: As the nature of the foundation model is not part of this lecture, feel free to ask TAs if you are curious).

Run the cell below to extract the provided train and test dataset. Each image is represented by a 768-d feature vector extracted from a visual foundation model. The train and test datasets contain feature vectors of 878 and 186 images respectively.

In [45]:
import torch 

# Label mapping
label_to_classname = {0 : "TUMOR", 1 : "STROMA"}

# Train features and labels
train_features = torch.load(os.path.join(data_base_path, data_folder, "part1/k16_train_features.pth"))
train_labels = torch.load(os.path.join(data_base_path, data_folder, "part1/k16_train_labels.pth"))

# Test features and labels
test_features = torch.load(os.path.join(data_base_path, data_folder, "part1/k16_test_features.pth"))
test_labels = torch.load(os.path.join(data_base_path, data_folder, "part1/k16_test_labels.pth"))

test_features.shape

torch.Size([186, 768])

**Task 1 (2.5 points)** Based on the training features (```train_features```) and training labels (```train_labels```), classify the test features (```test_features```) using minimum Mahalanobis distance.

*Note:* You are not allowed to use any prebuilt Mahalanobis distance function. Additionally, ```torch.cov``` is not defined to compute the covariance matrix. You can use ```sklearn.covariance.LedoitWolf``` instead.

In [46]:
### Task 1
### YOUR CODE

**Task 2 (0.5 points)** Compute the accuracy of your predictions with the test labels (```test_labels```).

In [47]:
### Task 2
### YOUR CODE

### 1.2 Out-of-Distribution detection with Mahalanobis distance (3 points)

You will note that the test you run above is not really realistic. Like the training set, it contains only the TUMOR and STROMA tissue types. Nevertheless, at test time, the other tissues (Label -1) are also present and cannot be filtered by hand. Moreover, they cannot be recognized by the model as they are out of the training distribution (It is the consequence of the laziness of the annotators ;)). For this reason, it is essential to filter them out. This task is called Out-of-Distribution (OoD) detection. 

A simple way to do OoD detection is to compute for every test example an OoD-ness score which should be low for In-Distribution (ID) examples and high for OoDs. Then we define a threshold from which every example with an OoD-ness lying above is discarded, and those lying below are forwarded to the model for prediction. An example of OoD-ness score is the minimum Mahalanobis distance.

Run the cell below to load a new test set containing OoD examples. It has 186 ID and 558 OoD examples.

In [48]:
label_to_classname_w_ood = {0 : "TUMOR", 1 : "STROMA", -1 : "OoD"}

# Test features and labels with OoD tissues
test_features_w_ood = torch.load(os.path.join(data_base_path, data_folder, "part1/k16_test2_features.pth"))
test_labels_w_ood = torch.load(os.path.join(data_base_path, data_folder,"part1/k16_test2_labels.pth"))

test_features_w_ood.shape

torch.Size([744, 768])

**Task 1 (0.5 point)** Why do you think the minimum Mahalanobis distance is a good OoD-ness score?

**Answer:**

**Task 2 (0.5 point)** Compute the minimum Mahalanobis distance for every test examples in ```test_features_w_ood``` with respect to the training features (```train_features```).

In [28]:
### Task 2
### YOUR CODE

**Task 3 (0.5 point)** Plot a histogram to show the difference between the Mahalanobis distance of TUMOR, STROMA and OoD tissue types and comment on what you observe.

In [29]:
### Task 3
### YOUR CODE

**Observations:**

**Task 4 (1 point)** Find a threshold on the Mahalanobis distance such that 95% of the OoD examples are filtered out. How much TUMOR and STROMA have also been filtered out?

In [30]:
### Task 4
### YOUR CODE

**Task 5 (0.5 point)** Assign prediction -1 to filtered out examples and compute the average class-wise accuracy of your prediction with test labels (```test_labels_w_ood```). Is it satisfactory?

In [31]:
### Task 5
### YOUR CODE

### 1.3 Out-of-distribution detection with k-NN classifier (6 points)

The visual foundation models are known to be very good k-NN classifiers. It motivates us to implement a k-NN classifier to recognize TUMOR and STROMA. Moreover, k-NN distance is a good OoD-ness score and suits our task.

**Task 1 (2 points)** Based on the training features (```train_features```) and training labels (```train_labels```), classify the test features (```test_features```) using a k-NN classifier. Then report the accuracy of your predictions with the test labels (```test_labels```).

*Note:* The choice of `k` is up to you.

In [32]:
from sklearn import metrics
import numpy as np

In [33]:
print(train_features.shape)
print("878 images in the train feature, with 768 features each.")

torch.Size([878, 768])
878 images in the train feature, with 768 features each.


In [34]:
print(train_labels.shape)
print("878 labels (0 or 1) corresponding to the train features")

(878,)
878 labels (0 or 1) corresponding to the train features


In [35]:
"""
TODO
"""

def kNN(neighbors, labels, k, vector):
    assert(k>0) # check if the k is valid
    
    # get the encludian distances between the vector and all the neighbors
    distances = metrics.pairwise.euclidean_distances(neighbors, vector.unsqueeze(0)) 
    distances = distances.reshape((distances.shape[0]))
    
    # get the k neighbors with the smallest distance to vector and check their labels
    arg_mins = np.argpartition(distances, k-1)
    closest_neighbor_indices = arg_mins[:k]
    closest_labels = labels[closest_neighbor_indices]
    
    # Get the label of vector according to the majority vote (with preference to 1 if tie in case of k even)
    zero_num = np.count_nonzero(closest_labels==0)
    if zero_num > k//2:
        return 0
    return 1

In [36]:
k = 100
test_labels_pred = [kNN(train_features, train_labels, k, test_vector) for test_vector in test_features]

In [42]:
acc = [0]*2
for i in range(len(test_labels)):
    acc[0] += 1 if test_labels[i] == 0 and test_labels_pred[i] == 1 else 0
    acc[1] += 1 if test_labels[i] == 1 and test_labels_pred[i] == 0 else 0
    
num_ones = sum(test_labels)
num_zeros = len(test_labels) - num_ones
acc[0] = acc[0] / num_zeros
acc[1] = acc[1] / num_ones

acc = 1-(acc[0] + acc[1])/2

print("The accuracy of the kNN is {acc:.2f} % with k = {k}".format(acc=acc*100,k=k))

The accuracy of the kNN is 99.46 % with k = 100


**Task 2 (2 points)** Perform OoD detection on the test features (```test_features_w_ood```) using a k-NN distance based OoD-ness score. Find a threshold on your OoD-ness score such that 95% of the OoD examples are filtered out. How much TUMOR and STROMA have also been filtered out? Finally, assign prediction -1 to filter out examples and compute the average class-wise accuracy of your prediction with test labels (```test_labels_w_ood```).

*Note:* The OoD-ness is based on the distance to the k-nearest neighbors. The formulation is up to you. You have to justify your choice.

In [43]:
"""
TODO
"""
def kNN_OoD_old(neighbors, labels, k, threshold, vector, k_2=100):
    assert(k>0) # check if the k is valid
    
    # get the encludian distances between the vector and all the neighbors and take the k smallest ones
    distances = metrics.pairwise.euclidean_distances(neighbors, vector.unsqueeze(0)) 
    distances = np.sort(distances.reshape((distances.shape[0])))
    knn_distances = distances[:k]
    
    # Take the mean of these knn distances and divide it by the max distance with all neighbors
    mean_knn_distance = np.mean(knn_distances)
    std = np.std(knn_distances)
    max_n_distance = distances[-1]
    ratio = mean_knn_distance/max_n_distance
    
    # Use this ratio to filter OoD
    if ratio > threshold:
        return -1 # It is OoD
    return kNN(neighbors, labels, max(1,k//4), vector) # it is not OoD

In [309]:
"""
TODO
"""
def kNN_OoD(neighbors, labels, k, vector, mean_thr=52, std_thr=2):
    assert(k>0) # check if the k is valid
    
    # get the encludian distances between the vector and all the neighbors and take the k smallest ones
    distances = metrics.pairwise.euclidean_distances(neighbors, vector.unsqueeze(0)) 
    distances = np.sort(distances.reshape((distances.shape[0])))
    knn_distances = distances[:k]
    
    # Take the mean of these knn distances and divide it by the max distance with all neighbors
    mean_knn_distance = np.mean(knn_distances)
    std = np.std(knn_distances)
    
    # Use this ratio to filter OoD
    if mean_knn_distance > mean_thr and std < std_thr:
        return -1 # It is OoD
    return kNN(neighbors, labels, max(1,k//4), vector) # it is not OoD

In [408]:
"""
High score for OoD
"""
def ood_score_knn(knn_distances):
    mean_knn_distance = np.mean(knn_distances)
    std_knn_distance = np.std(knn_distances)
    ratio = mean_knn_distance / std_knn_distance
    return ratio

In [427]:
"""
TODO
"""
def get_kNN_ood_score(neighbors, labels, k, vectors, thr=0.95):
    assert(k>0) # check if the k is valid
    
    ood_num = np.count_nonzero(labels==-1)
    print(ood_num)
    
    score_indexed = []

    for i in range(len(vectors)):
        vector = vectors[i]
        # get the encludian distances between the vector and all the neighbors and take the k smallest ones
        distances = metrics.pairwise.euclidean_distances(neighbors, vector.unsqueeze(0)) 
        distances = np.sort(distances.reshape((distances.shape[0])))
        knn_distances = distances[:k]

        score_indexed.append((ood_score_knn(knn_distances), i))
    
    score_indexed = sorted(score_indexed)
    true_ood_evicted = 0
    for score, index in score_indexed:
        true_ood_evicted += 1 if labels[index] == -1 else 0
        if true_ood_evicted/ood_num >= 0.95:
            return score

In [428]:
k = 30
get_kNN_ood_score(train_features, train_labels, k, test_features_w_ood, )

ZeroDivisionError: division by zero

In [44]:
"""
TODO
"""
def kNN_analysis(neighbors, labels, k, threshold, vector, label, p=False):
    assert(k > 0) # check if the k is valid
    
    # get the encludian distances between the vector and all the neighbors and take the k smallest ones
    distances = metrics.pairwise.euclidean_distances(neighbors, vector.unsqueeze(0)) 
    distances = np.sort(distances.reshape((distances.shape[0])))
    knn_distances = distances[:k]
    
    # Take the mean of these knn distances and divide it by the max distance with all neighbors
    mean_knn_distance = np.mean(knn_distances)
    std = np.std(knn_distances)
    ratio = mean_knn_distance * (1/std)
    
    ood_knn_thr_ratio = np.percentile(ratio[labels == -1], 100 -threshold * 100)
    
      
    return mean_knn_distance, std, ratio

In [397]:
k = 30
kNN_analysis(train_features, train_labels, k, threshold=0.95, )

In [ ]:
acc = [0]*3
for i in range(len(test_labels)):
    acc[0] += 1 if test_labels[i] == 0 and test_labels_pred[i] != 0 else 0
    acc[1] += 1 if test_labels[i] == 1 and test_labels_pred[i] != 1 else 0
    acc[2] += 1 if test_labels[i] == -1 and test_labels_pred[i] != -1 else 0
    
num_ones = sum(1 for i in test_labels if i == 1)
num_m_ones = sum(1 for i in test_labels if i == -1)
num_zeros = sum(1 for i in test_labels if i == 0)
acc[0] = acc[0] / num_zeros
acc[1] = acc[1] / num_ones
acc[2] = acc[2] / num_m_ones

acc = 1-(sum(acc))/len(acc)

print("The accuracy of the kNN is {acc:.2f} % with k = {k}".format(acc=acc*100,k=k))

In [398]:
# LABEL 0
n_0 = 93
mean_0 = []
std_0 = []
ratio_0 = []
for i in range(n_0):
    test_vector_w_ood = test_features_w_ood[i]
    test_label_w_ood = test_labels_w_ood[i]
    m,s,r=kNN_analysis(train_features,train_labels, k, threshold, test_vector_w_ood, test_label_w_ood)
    mean_0.append(m)
    std_0.append(s)
    ratio_0.append(r)
mean_0 = np.mean(mean_0)
std_0 = np.mean(std_0)
ratio_0 = np.mean(ratio_0)
print("For label 0, mean:",mean_0,"and std:",std_0,"and ratio:",ratio_0)

For label 0, mean: 50.08739 and std: 2.3572955 and ratio: 23.337691674364308


In [399]:
# LABEL 1
n_1 = 186
mean_1 = []
std_1 = []
ratio_1 = []
for i in range(n_0,n_1):
    test_vector_w_ood = test_features_w_ood[i]
    test_label_w_ood = test_labels_w_ood[i]
    m, s, r = kNN_analysis(train_features,train_labels, k, threshold, test_vector_w_ood, test_label_w_ood)
    mean_1.append(m)
    std_1.append(s)
    ratio_1.append(r)
mean_1 = np.mean(mean_1)
std_1 = np.mean(std_1)
ratio_1 = np.mean(ratio_1)
print("For label 1, mean:",mean_1,"and std:",std_1,"and ratio:",ratio_1)

For label 1, mean: 48.81031 and std: 3.0706406 and ratio: 17.637571631690154


In [400]:
# LABEL -1
n_m1 = len(test_features_w_ood)
mean_m1 = []
std_m1 = []
ratio_m1 = []
for i in range(n_1,n_m1):
    test_vector_w_ood = test_features_w_ood[i]
    test_label_w_ood = test_labels_w_ood[i]
    m, s, r = kNN_analysis(train_features,train_labels, k, threshold, test_vector_w_ood, test_label_w_ood)
    mean_m1.append(m)
    std_m1.append(s)
    ratio_m1.append(r)

mean_m1 = np.mean(mean_m1)
std_m1 = np.mean(std_m1)
ratio_m1 = np.mean(ratio_m1)
print("For label -1, mean:",mean_m1,"and std:",std_m1,"and ratio:",ratio_m1)

For label -1, mean: 62.146156 and std: 1.8802456 and ratio: 37.504309696645805


In [377]:
test_labels_w_ood_pred = [kNN_OoD(train_features, train_labels, k, test_vector, mean_thr=50, std_thr=3) for test_vector in test_features_w_ood]

In [388]:
print(test_labels_w_ood_pred)
print(test_labels_w_ood)

[-1, -1, 0, -1, 0, 0, -1, -1, -1, 0, 0, -1, -1, -1, -1, -1, -1, -1, 0, 0, -1, -1, -1, -1, 0, -1, -1, -1, -1, -1, 0, -1, 0, -1, 0, 0, 0, -1, -1, -1, -1, 0, -1, 0, 0, -1, -1, -1, -1, -1, -1, 0, -1, -1, -1, -1, -1, 0, 0, -1, -1, 0, 0, -1, -1, -1, -1, 0, -1, -1, 0, -1, -1, -1, 0, -1, -1, -1, 0, -1, -1, -1, -1, -1, 0, 0, 0, -1, -1, 0, -1, 0, 0, -1, -1, 1, 1, -1, 1, 1, -1, -1, 1, -1, -1, 1, 1, 1, -1, -1, -1, -1, 1, 1, 1, -1, 1, 1, -1, 1, 1, -1, 1, 1, 1, 1, 1, -1, -1, -1, 1, -1, 1, 1, 1, 1, -1, 1, 1, 1, 1, 1, 1, -1, 1, -1, -1, 1, -1, 1, 1, -1, 1, -1, -1, -1, -1, 1, 1, -1, -1, 1, 1, 1, 1, 1, -1, -1, -1, 1, -1, 1, -1, -1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, -1, 1, -1, 1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 1, -1, -1, -1, -1, -1, 1, -1, -1, 1, -1, -1, -1, -1, -1, -1, -1, -1, 1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 1, -1, -1, -1, -1, 

In [378]:
OoD_filtered_out = 0
OoD_num = np.count_nonzero(test_labels_w_ood==-1)
for i in range(len(test_labels_w_ood)):
    OoD_filtered_out += 1 if test_labels_w_ood[i] == -1 and test_labels_w_ood_pred[i] == -1 else 0
OoD_filtered_out = OoD_filtered_out / OoD_num

print("The percentage of the OoD filtered out is {perc:.2f} % with k = {k} and threshold = {thr}".format(perc=OoD_filtered_out*100,k=k,thr=threshold))

The percentage of the OoD filtered out is 94.80 % with k = 20 and threshold = 0.75


In [379]:
acc = 0
OoD_num = np.count_nonzero(test_labels_w_ood==-1)
for i in range(len(test_labels_w_ood)):
    if test_labels_w_ood[i] != -1:
        acc += 1 if test_labels_w_ood[i] == test_labels_w_ood_pred[i] else 0
acc = acc/(len(test_labels_w_ood)-OoD_num)

print("The accuracy of the kNN is {acc:.2f} % with k = {k} and threshold = {thr}".format(acc=acc*100,k=k,thr=threshold))

The accuracy of the kNN is 46.77 % with k = 20 and threshold = 0.75


**Task 3 (1 point)** Is k-NN better than Mahalanobis distance ? Make an hypothesis for the reasons.

**Answer:**

**Task 4 (1 point)** Do you think we can suggest the approach presented in this exercise to compute TUMOR/STROMA ratio automatically ? Justify your thoughs. If not, suggest at least two ideas to improve it.

*Note:* Annotating all the training dataset is not an option.

**Answer:**

---

## Part 2 (12 points)
In this part, we aim to classify cervical cells resulting from Pap smear tests. To that end we'll be using a publicly available cell dataset: Sipakmed (https://www.cs.uoi.gr/~marina/sipakmed.html). The dataset is composed of 4049 images of isolated cells cropped from 966 cluster cell images of Pap smear slides. Each cell in the dataset has been categorized in either of the following categories: 

    - Superficial-Intermediate.
    - Parabasal.
    - Koilocytotic.
    - Dysketarotic.
    - Metaplastic.
Your objective is to implement a classifier to automate the cell classification process. To ease your work we provide you with pre-computed embeddings for each images (`lab-03-data/part2/sipakmed_clean_embeddings.pth`). The embeddings are obtained from a pre-trained ResNet-50 (https://arxiv.org/pdf/1512.03385.pdf) and the corresponding images are also provided (`lab-03-data/part2/sipakmed_clean`). Note that you are free to discard the provided embeddings and work directy with the images.

### 2.1 Dataset (4 points)
Your first task is prepare the dataset such that it can be used to train your model. For that purpose we prepared the skeleton of the class `Sipakmed` that inherits from the class `Dataset` of PyTorch. Read the documentation (https://pytorch.org/tutorials/beginner/basics/data_tutorial.html#creating-a-custom-dataset-for-your-files) and complete the missing parts.

In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from PIL import Image
import os
from torch import nn
from collections import OrderedDict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import math

In [ ]:
# Load the features
features_path = '../data/lab-03-data2023/part2/sipakmed_clean_embeddings.pth'

In [ ]:
class Sipakmed(Dataset):
    phase_dict = {
            'train': {'start': 0.0, 'stop': 0.5},
            'val': {'start': 0.5, 'stop': 0.75},
            'test': {'start': 0.75, 'stop': 1.0}
    }
    label_dict = {
        'im_Superficial-Intermediate': 0,
        'im_Parabasal': 1, 
        'im_Metaplastic': 2,
        'im_Koilocytotic': 3,
        'im_Dyskeratotic': 4
    }
    
    def __init__(self, features_path, phase):

        super(Sipakmed, self).__init__()
        # Store class attributes
        self.phase = phase
        
        # Collect the dataimport torch
        import torch.nn.functional as F
        import numpy as np
        self.raw_data = torch.load(features_path)
        self.features, self.labels, self.paths = self.collect_data()
        
    def collect_data(self):
        # Iterate over the dirs/classes
        features, labels, paths = [], [], []
        for dir_name, dir_dict in self.raw_data.items():
            # Get the paths and embeddings
            dir_paths, dir_embeddings = list(zip(*[(k, v) for k, v in dir_dict.items()]))
            
            # Split
            n = len(dir_paths)
            np.random.seed(42)
            permutations = np.random.permutation(n)
            dir_paths = np.array(dir_paths)[permutations]
            dir_embeddings = torch.stack(dir_embeddings)[permutations]
            n_start = int(n * self.phase_dict[self.phase]['start'])
            n_stop = int(n * self.phase_dict[self.phase]['stop'])
            dir_embeddings = dir_embeddings[n_start: n_stop]
            dir_paths = dir_paths[n_start: n_stop]
    
            # Store
            features.append(dir_embeddings)
            paths.append(dir_paths)
            dir_labels = torch.tensor([self.label_dict[p.split('/')[-2]] for p in dir_paths])
            labels.append(dir_labels)
            
        # Merge
        features = torch.cat(features)
        labels = torch.cat(labels)
        paths = np.concatenate(paths)
        return features, labels, paths
            
        
    def __len__(self,):
        """
        Returns the number of samples in the dataset.
        """
        ### YOUR CODE
    
    def __getitem__(self, index):
        """
        Returns the embedding, label, and image path of queried index.
        """
        ### YOUR CODE
        return embedding, label, path

Once the implementation of `Sipakmed` completed, create 3 instances of the class (train/val/test) with the corresponding `phase` flag.

In [ ]:
# Instantiate the datasets
train_dataset = ### YOUR CODE
val_dataset = ### YOUR CODE
test_dataset = ### YOUR CODE

Now that your datasets are ready, use the class `DataLoader` from PyTorch to let it handle efficiently the batching, shuffling, etc. of your data.

In [ ]:
# Instantiate the data loaders
train_loader = ### YOUR CODE
val_loader = ### YOUR CODE
test_loader = ### YOUR CODE

Get to know your data. Plot a few example images for each class of your dataset.

In [ ]:
# Visualize some training example
### YOUR CODE

### 2.2 Training (4 points)
In this part your objective is to implement the required tools to train your model. The first thing you'll need is a a model which takes as input the pre-computed features and returns the corresponding class probabilities/logits.

In [ ]:
# Implement the model
embedding_dim = train_dataset.features.shape[1]
model = ### YOUR CODE

The optimizer will keep track of your model's parameters, gradients, etc (https://pytorch.org/docs/stable/optim.html). It is responsible to update your model's parameters after each forward pass using the backpropagation algorithm.

In [ ]:
# Set the optimizer
optimizer = ### YOUR CODE

In [ ]:
# Set the loss
criterion = ### YOUR CODE

Implement a function that takes as input the model's output and the corresponding labels and returns the perçentage of correct predictions.

In [ ]:
def accuracy(outputs, labels):
    """
    Computes the accuracy of predictions based on the model outputs (NxK: N samples, K classes) 
    and the labels (N: N samples).
    """
    ### YOUR CODE

Implement a funtion `train` that forwards the complete training set through your model (= 1 epoch) and updates its parameters after each forward pass. To keep track of the training process make sure to at least return the accuracy of the model and the average loss it incurred through the current epoch.

In [ ]:
def train(model, optimizer, criterion, loader):
    # Set the model in train mode
    ### YOUR CODE
    
    # Iterate over the batches
    full_outputs = []
    full_labels = []
    losses = []
    for batch in loader:
        # Get the embeddings, labels and paths 
        ### YOUR CODE
        
        # Feed the embeddings to the model
        ### YOUR CODE

        # Compute cross entropy loss
        ### YOUR CODE
        
        # Reset the gradients
        ### YOUR CODE
        
        # Backpropagate
        ### YOUR CODE

        # Update the parameters
        ### YOUR CODE
        
        # Store the outputs, labels and loss
        ### YOUR CODE
    
    # Concat
    full_outputs = torch.cat(full_outputs).cpu()
    full_labels = torch.cat(full_labels).cpu()
    losses = torch.stack(losses).mean().cpu()
    
    # Compute the accuracy
    ### YOUR CODE
    return acc, full_outputs, full_labels, losses

Implement a funtion `validate` that forwards the complete validation or test set through your model and evaluates its predictions. To keep track of the training process make sure to at least return the accuracy of the model and the average loss it incurred through the current epoch.

In [ ]:
@torch.no_grad()
def validate(model, criterion, loader):
    # Set the model in train mode
    ### YOUR CODE
    
    # Iterate over the batches
    full_outputs = []
    full_labels = []
    full_paths = []
    losses = []
    for batch in loader:
        # Get the embeddings, labels and paths
        ### YOUR CODE
        
        # Feed the embeddings to the model
        ### YOUR CODE

        # Compute cross entropy loss
        l### YOUR CODE
        
        # Store the outputs, labels and loss
        ### YOUR CODE
    
    # Concat
    full_outputs = torch.cat(full_outputs).cpu()
    full_labels = torch.cat(full_labels).cpu()
    losses = torch.stack(losses).mean().cpu()
    full_paths = np.concatenate(full_paths)
    
    # Compute the accuracy
    ### YOUR CODE
    return acc, full_outputs, full_labels, losses, full_paths

You should now be able to train you model. Alternate between training and validation steps to find and save the best model (best accuracy on the validation set).

In [ ]:
# Main loop
epochs = ### YOUR CODE
best_acc = ### YOUR CODE
model_savepath = '../data'

for epoch in range(epochs):
    # Train
    ### YOUR CODE

    # Evaluate
    ### YOUR CODE
    
    # Save the model
    if val_acc > best_acc:
        ### YOUR CODE

### 2.3 Evaluation (4 points)
Re-load the best model and evaluate its predictions on the test set.

In [ ]:
# Re-load the best model
### YOUR CODE

# Evaluate
### YOUR CODE

A useful tool to analyze your model's performance on the different classes is the confusion matrix (https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html). Computes its entries for your model and the test set.

In [ ]:
# Display the confusion matrix
### YOUR CODE

Alternatively it can be useful to plot the problematic samples as well as the predicted and ground truth classes. Can you do so?

In [ ]:
# Find the misclassified samples
### YOUR CODE

# Plot the misclassified samples
### YOUR CODE